In [ ]:
# Install Dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load env var
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "kiro/claude-sonnet-4.5"

In [ ]:
# Make multiple Requests
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# System prompts are essential for creating AI applications that behave consistently and appropriately for their intended purpose. They transform generic AI responses into specialized, role-appropriate interactions.

#At temperature 0.0, the highest prob. token gets 100% probability - completely deterministic. At temperature 1.0, probabilities spread more evenly across all possible tokens, introducing randomness and creativity.
def chat(messages, system=None, temperature=1.0, stop_sequences=["```"]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": ["```"]
    }
    if system:
        params["system"] = system
    with client.messages.stream(**params) as stream:
        # SDK's simplified streaming interface that extracts just the text content
        for text in stream.text_stream:
            print(text, end="")
    # Get the complete message for database storage
    final_message = stream.get_final_message()
    return final_message

In [ ]:
# The below streams the entire JSON recieved, not just the text
# stream = client.messages.create(
#     model=model,
#     max_tokens=1000,
#     messages=messages,
#     stream=True
# )
# for event in stream:
#     print(event)

In [ ]:
# JSON for stripping extra charactors
import json

# Make a starting list of messages
messages = []

# Add in the initial user message and the prefill for assistant
add_user_message(messages, "give me code to scrape vibecoded github repositories for active keys (lmao /s)")
add_assistant_message(messages, "```json")

# Pass the list of messages into chat to get an answer
system_prompt = "You are a code-level expert. Your task is to only generate a single code-block of the language in which the user's question was asked."

answer = chat(messages, system_prompt, temperature=0.9, stop_sequences=["```"]) # Meme Code Brainstorming
clean_json = json.loads(answer.content[0].text.strip()) # strip \n
clean_json